In [73]:
import transformers
import torch
import pandas as pd
import nltk
from nltk.corpus import brown
from collections import Counter

In [27]:
data = pd.read_csv("mtsamples.csv")

In [28]:
print(data.head(10))

   Unnamed: 0                                        description  \
0           0   A 23-year-old white female presents with comp...   
1           1           Consult for laparoscopic gastric bypass.   
2           2           Consult for laparoscopic gastric bypass.   
3           3                             2-D M-Mode. Doppler.     
4           4                                 2-D Echocardiogram   
5           5   Morbid obesity.  Laparoscopic antecolic anteg...   
6           6   Liposuction of the supraumbilical abdomen, re...   
7           7                                 2-D Echocardiogram   
8           8   Suction-assisted lipectomy - lipodystrophy of...   
9           9                         Echocardiogram and Doppler   

             medical_specialty                                sample_name  \
0         Allergy / Immunology                         Allergic Rhinitis    
1                   Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                   

In [29]:
print(data.columns)

Index(['Unnamed: 0', 'description', 'medical_specialty', 'sample_name',
       'transcription', 'keywords'],
      dtype='object')


In [30]:
len(data)

4999

In [35]:
data = data.dropna().reset_index(drop=True)

In [36]:
len(data)

3898

In [37]:
transcription_data = data["transcription"]

In [38]:
print(transcription_data)

0       SUBJECTIVE:,  This 23-year-old white female pr...
1       PAST MEDICAL HISTORY:, He has difficulty climb...
2       HISTORY OF PRESENT ILLNESS: , I have seen ABC ...
3       2-D M-MODE: , ,1.  Left atrial enlargement wit...
4       1.  The left ventricular cavity size and wall ...
                              ...                        
3893    ADMISSION DIAGNOSIS:,  Morbid obesity.  BMI is...
3894    HISTORY OF PRESENT ILLNESS:,  Ms. A is a 55-ye...
3895    PAST MEDICAL HISTORY:  ,She had a negative str...
3896    HISTORY:,  A 55-year-old female presents self-...
3897    ADMITTING DIAGNOSIS: , Kawasaki disease.,DISCH...
Name: transcription, Length: 3898, dtype: object


In [39]:
import joblib

In [40]:
# Using to_csv so the file is readable in Excel
transcription_data.to_csv("m_t_data.csv", index=False)

In [41]:
from google.colab import files

# Assuming you want to download the CSV file you just created
files.download('m_t_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
import re

In [56]:
def process_data(file_name):
    words = []
    # Read the CSV file
    df = pd.read_csv(file_name)
    content = " ".join(df.iloc[:, 0].dropna().astype(str).tolist())
    content = content.lower()
    # Find all words using regex
    words = re.findall(r'\w+', content)
    return words

In [64]:
word_l = process_data('m_t_data.csv')
vocab = set(word_l)
print(f"The first ten words in the text are: \n{word_l[1:1000]}")
print(f"There are {len(vocab)} unique words in the vocabulary.")

The first ten words in the text are: 
['this', '23', 'year', 'old', 'white', 'female', 'presents', 'with', 'complaint', 'of', 'allergies', 'she', 'used', 'to', 'have', 'allergies', 'when', 'she', 'lived', 'in', 'seattle', 'but', 'she', 'thinks', 'they', 'are', 'worse', 'here', 'in', 'the', 'past', 'she', 'has', 'tried', 'claritin', 'and', 'zyrtec', 'both', 'worked', 'for', 'short', 'time', 'but', 'then', 'seemed', 'to', 'lose', 'effectiveness', 'she', 'has', 'used', 'allegra', 'also', 'she', 'used', 'that', 'last', 'summer', 'and', 'she', 'began', 'using', 'it', 'again', 'two', 'weeks', 'ago', 'it', 'does', 'not', 'appear', 'to', 'be', 'working', 'very', 'well', 'she', 'has', 'used', 'over', 'the', 'counter', 'sprays', 'but', 'no', 'prescription', 'nasal', 'sprays', 'she', 'does', 'have', 'asthma', 'but', 'doest', 'not', 'require', 'daily', 'medication', 'for', 'this', 'and', 'does', 'not', 'think', 'it', 'is', 'flaring', 'up', 'medications', 'her', 'only', 'medication', 'currently', '

In [65]:
def get_count(word_l):
    word_count_dict = {}

    for w in word_l:
        word_count_dict[w] = word_count_dict.get(w,0)+1

    return word_count_dict

In [71]:
word_count_dict = get_count(word_l)
print(f"There are {len(word_count_dict)} key values pairs")
print(f"The count for the word 'and' is {word_count_dict.get('and',0)}")
print(f"The count for the word 'he' is {word_count_dict.get('he',0)}")

There are 18867 key values pairs
The count for the word 'and' is 58799
The count for the word 'he' is 7738


In [68]:
def get_probs(word_count_dict):
    probs = {}
    ### START CODE HERE ###
    total_count = sum(word_count_dict.values())
    # get the total count of words for all words in the dictionary
    for k in word_count_dict:
        count = word_count_dict.get(k,0)
        probability = count/total_count
        probs[k] = probability
    ### END CODE HERE ###
    return probs

In [72]:
probs = get_probs(word_count_dict)
print(f"Length of probs is {len(probs)}")
print(f"P('and') is {probs['and']:.4f}")
print(f"P('he') is {probs['he']:.4f}")

Length of probs is 18867
P('and') is 0.0354
P('he') is 0.0047


<h2>Merge nltk brown corpus to my vocab</h2>

In [74]:
nltk.download('brown')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [75]:
# Get general English words from Brown corpus, lowercase them
brown_words = [word.lower() for word in brown.words() if word.isalpha()]

In [76]:
# Build frequency counters for both sources
medical_freq = Counter(vocab)        # 'words' = your list from process_data_csv()

In [77]:
print(medical_freq)

Counter({'slit': 1, 'amalgam': 1, 'hx': 1, 'equagesic': 1, 'integrilin': 1, 'term': 1, 'fosamax': 1, 'rr13': 1, 'ahmed': 1, 'brachytherapy': 1, 'floppy': 1, 'phlebotomies': 1, 'lay': 1, 'reassess': 1, 'barbecues': 1, 'partum': 1, 'head': 1, 'diastolic': 1, 'uncinate': 1, 'sentences': 1, 'hemostased': 1, 'myotendinous': 1, 'missouri': 1, 'postbronchodilator': 1, 'vegetation': 1, 'afo': 1, 'august': 1, 'intracapsular': 1, 'raised': 1, 'indicative': 1, 'elavil': 1, 'methylprednisolone': 1, 'neuritic': 1, 'comorbidities': 1, 'nasolabial': 1, 'quadrilateral': 1, 'rt': 1, 'vasculitis': 1, 'foveal': 1, 'deer': 1, 'pronunciation': 1, 'sent': 1, 'dissection': 1, 'orifice': 1, 'transcutaneous': 1, '140s': 1, 'hypoxic': 1, 'participating': 1, 'dissolution': 1, 'laparoscopically': 1, 'neighborhood': 1, 'excited': 1, 'windows': 1, 'ascites': 1, 'sweets': 1, 'photographic': 1, 'yards': 1, 'peritomy': 1, 'feasibility': 1, 'deferens': 1, 'downfractured': 1, 'hemangioma': 1, 'blackman': 1, '6100': 1, '

In [78]:
general_freq = Counter(brown_words)

In [79]:
print(general_freq)

Counter({'the': 69971, 'of': 36412, 'and': 28853, 'to': 26158, 'a': 23195, 'in': 21337, 'that': 10594, 'is': 10109, 'was': 9815, 'he': 9548, 'for': 9489, 'it': 8760, 'with': 7289, 'as': 7253, 'his': 6996, 'on': 6741, 'be': 6377, 'at': 5372, 'by': 5306, 'i': 5164, 'this': 5145, 'had': 5133, 'not': 4610, 'are': 4394, 'but': 4381, 'from': 4370, 'or': 4206, 'have': 3942, 'an': 3740, 'they': 3620, 'which': 3561, 'one': 3292, 'you': 3286, 'were': 3284, 'her': 3036, 'all': 3001, 'she': 2860, 'there': 2728, 'would': 2714, 'their': 2669, 'we': 2652, 'him': 2619, 'been': 2472, 'has': 2437, 'when': 2331, 'who': 2252, 'will': 2245, 'more': 2215, 'if': 2198, 'no': 2139, 'out': 2097, 'so': 1985, 'said': 1961, 'what': 1908, 'up': 1890, 'its': 1858, 'about': 1815, 'into': 1791, 'than': 1790, 'them': 1788, 'can': 1772, 'only': 1748, 'other': 1702, 'new': 1635, 'some': 1618, 'could': 1601, 'time': 1598, 'these': 1573, 'two': 1412, 'may': 1402, 'then': 1380, 'do': 1363, 'first': 1361, 'any': 1344, 'my': 

In [80]:
# Merge: Counter supports direct addition, summing counts for shared words
merged_freq = medical_freq + general_freq

In [81]:
print(merged_freq)

Counter({'the': 69972, 'of': 36413, 'and': 28854, 'to': 26159, 'a': 23196, 'in': 21338, 'that': 10595, 'is': 10110, 'was': 9816, 'he': 9549, 'for': 9490, 'it': 8761, 'with': 7290, 'as': 7254, 'his': 6997, 'on': 6742, 'be': 6378, 'at': 5373, 'by': 5307, 'i': 5165, 'this': 5146, 'had': 5134, 'not': 4611, 'are': 4395, 'but': 4382, 'from': 4371, 'or': 4207, 'have': 3943, 'an': 3741, 'they': 3621, 'which': 3562, 'one': 3293, 'you': 3287, 'were': 3285, 'her': 3037, 'all': 3002, 'she': 2861, 'there': 2729, 'would': 2715, 'their': 2670, 'we': 2653, 'him': 2620, 'been': 2473, 'has': 2438, 'when': 2332, 'who': 2253, 'will': 2246, 'more': 2216, 'if': 2199, 'no': 2140, 'out': 2098, 'so': 1986, 'said': 1962, 'what': 1909, 'up': 1891, 'its': 1859, 'about': 1816, 'into': 1792, 'than': 1791, 'them': 1789, 'can': 1773, 'only': 1749, 'other': 1703, 'new': 1636, 'some': 1619, 'could': 1602, 'time': 1599, 'these': 1574, 'two': 1413, 'may': 1403, 'then': 1381, 'do': 1364, 'first': 1362, 'any': 1345, 'my': 

In [82]:
import pickle
with open("merged_vocab.pkl", "wb") as f:
    pickle.dump(merged_freq, f)

In [83]:
from google.colab import files

# Assuming you want to download the CSV file you just created
files.download('merged_vocab.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>